# Unsupervised HP search + multi-seed finals

Hyperparameters are selected by **held-out unsupervised metrics** (Bernoulli LL for LV/GNN; negated row MSE for PCA). Ground-truth Hungarian / ARI / NMI are reported only on multi-seed final runs.

Expected artifacts from `run_experiments.py` / `launch_lightning_sweep.py`:
- `hp_results.csv`, `hp_best.json`
- `final_results.csv`, `final_summary.csv`

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 120
PALETTE = {"gnn": "#2a6f97", "lv": "#e76f51", "pca": "#6c757d"}

## 1. Hyperparameter search (unsupervised selection)

In [ ]:
hp = pd.read_csv("hp_results.csv")
best = json.loads(Path("hp_best.json").read_text())
display(hp.sort_values(["method", "val_metric"], ascending=[True, False]).groupby("method").head(5))
print("Selected by val_metric:")
display(pd.DataFrame([
    {"method": m, "name": v["name"], "val_metric": v["val_metric"], **v["hyperparams"]}
    for m, v in best.items()
]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=False)
for ax, method in zip(axes, ["pca", "lv", "gnn"]):
    sub = hp[hp["method"] == method].copy()
    if sub.empty:
        ax.set_visible(False)
        continue
    sns.scatterplot(
        data=sub, x="val_metric", y="gt_hungarian",
        hue="d", style="lr", ax=ax, palette="viridis",
    )
    ax.set_title(f"{method}: val vs GT (GT not used for selection)")
    ax.set_xlabel("held-out val_metric")
    ax.set_ylabel("Hungarian (GT)")
plt.tight_layout()
plt.show()

## 2. Multi-seed uncertainty (final runs)

In [ ]:
final = pd.read_csv("final_results.csv")
summary = pd.read_csv("final_summary.csv")
display(summary)
display(final.sort_values(["method", "seed"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = ["pca", "lv", "gnn"]
sns.boxplot(
    data=final, x="method", y="gt_hungarian", order=order,
    hue="method", palette=PALETTE, legend=False, ax=ax,
)
sns.stripplot(
    data=final, x="method", y="gt_hungarian", order=order,
    color="black", alpha=0.6, ax=ax,
)
ax.set_ylabel("Hungarian vs visual types")
ax.set_title("Uncertainty over final seeds (best HP per method)")
plt.tight_layout()
plt.show()

for _, row in summary.iterrows():
    print(
        f"{row['method']}: Hungarian {row['hungarian_mean']:.1f} ± {row['hungarian_std']:.1f} "
        f"(ARI {row['ari_mean']:.3f} ± {row['ari_std']:.3f}, "
        f"NMI {row['nmi_mean']:.3f} ± {row['nmi_std']:.3f})"
    )